# Notebook 04: Claude Labels and Complaint Trends

**Team:** Teenna Pachika and Johanna Garcia

In this notebook we use Claude to label all the negative reviews, not just a small sample like in Phase 2. Then we check if Claude's labels make sense compared to our keyword rules, and look at which complaints went up or down over time and which ones other users agreed with the most. We save every chart in the `figures/` folder so we can use them in our slides.

In [1]:
# Importing libraries and our shared code from the src folder
import sys
sys.path.append("..")

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.preprocessing import load_reviews, get_negative
from src.app_versions import get_release_dates

# Loading the data and keeping only the negative reviews
df = load_reviews()
neg = get_negative(df).reset_index(drop=True)

# Getting the app update dates (same function Johanna uses, so our charts match)
release_dates = get_release_dates(df)

# Using the same red as the other notebooks
NEG_COLOR = "#B5544C"

# Making sure the figures folder exists so we can save charts for the slides
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

# Defining a function to save and show a chart
def save_chart(name):
    """Save the current chart to the figures folder and show it."""
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

# Checking the counts
print(f"All reviews: {len(df):,} | Negative reviews (1-2 stars): {len(neg):,}")
print("App updates found:", len(release_dates))

All reviews: 50,000 | Negative reviews (1-2 stars): 4,147
App updates found: 5


## 1. Labeling All the Negative Reviews with Claude

In Phase 2 we only tested Claude on 8 reviews. Our professor suggested labeling all of them since it's cheap, so that's what we're doing here. We're using the same prompt as our Phase 2 test (version 2), which already lets a review have more than one complaint and has an "other" category.

In [2]:
# Importing tools for the Claude API
import anthropic
from dotenv import load_dotenv

# Loading the API key from the .env file in the main project folder
load_dotenv("../.env", override=True)
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Using the same small, cheap model as Phase 2
CLAUDE_MODEL = "claude-haiku-4-5"

# Our prompt from Phase 2 (version 2)
PROMPT = """You are labeling negative Duolingo app reviews.
Categories:
- energy_hearts: running out of energy or hearts, limits on how much you can practice
- streak: losing or protecting a streak
- ads: too many ads or ads interrupting lessons
- paywall: having to pay, Super/Max/premium, subscription, price, features locked behind payment
- ai: AI features or Duolingo using AI instead of people
- icon_notifications: the changing app icon or pushy notifications
- bugs: crashes, glitches, things not working
- language_access: missing languages, or users in some countries (like Iran) losing access to features
- teaching_quality: lessons that are wrong, confusing, or don't actually teach well
- other: none of the above

A review can have more than one category. Use "other" only if nothing else fits.
Return ONLY a JSON list, with no other text, like: [{{"id": 0, "labels": ["streak"]}}]

Reviews:
{reviews}"""

# Keeping track of tokens so we can report the cost
tokens = {"in": 0, "out": 0}

# Defining a function to send a batch of reviews to Claude in one call
def label_with_claude(texts, prompt=PROMPT, model=CLAUDE_MODEL):
    """Send numbered reviews to Claude in one call and return a list of labels for each review."""
    numbered = "\n".join(f"{i}: {t}" for i, t in enumerate(texts))
    response = client.messages.create(
        model=model, max_tokens=2000,
        messages=[{"role": "user", "content": prompt.format(reviews=numbered)}],
    )
    raw = response.content[0].text.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    tokens["in"] += response.usage.input_tokens
    tokens["out"] += response.usage.output_tokens
    labels = {item["id"]: item["labels"] for item in json.loads(raw)}
    return [labels.get(i, ["missing"]) for i in range(len(texts))]

print("Claude is ready")

Claude is ready


In [3]:
# Testing Claude on 100 random negative reviews first (5 batches of 20), to check the labels and the cost
BATCH_SIZE = 20
test = neg.sample(100, random_state=42).copy()

test_labels = []
for start in range(0, len(test), BATCH_SIZE):
    batch = test["content"].iloc[start:start + BATCH_SIZE].tolist()
    test_labels += label_with_claude(batch)

test["claude_labels"] = test_labels

# Estimating the cost (Haiku 4.5 is about $1 per million input tokens and $5 per million output tokens)
cost = tokens["in"] / 1_000_000 * 1 + tokens["out"] / 1_000_000 * 5
print(f"Tokens used: {tokens['in']:,} in, {tokens['out']:,} out | Cost so far: ${cost:.3f}")
print(f"Estimated cost for all {len(neg):,} reviews: ${cost * len(neg) / len(test):.2f}")
print("Reviews Claude skipped:", sum(labels == ["missing"] for labels in test["claude_labels"]))

# Showing the first 10 so we can read them
test[["content", "claude_labels"]].head(10)

Tokens used: 6,005 in, 1,802 out | Cost so far: $0.015
Estimated cost for all 4,147 reviews: $0.62
Reviews Claude skipped: 0


,content,claude_labels
149,I boycott this app until you activate Duolingo...,[language_access]
1971,ai slop,[ai]
238,"Shame on you, not supporting Persian language ...",[language_access]
3956,its was very fun but i told him i was a begine...,[teaching_quality]
308,it's not completely free 🥺,[paywall]
1255,I didn't know why my mother's gcash acc got in...,[paywall]
4070,These bastards deducted Rs5000 from my account...,[paywall]
1261,good,[other]
70,Whilst the content itself is fine if you buy a...,"[paywall, bugs]"
2053,The energy system really ruined the app for me...,[energy_hearts]


In [4]:
# Labeling all the negative reviews, or loading the saved labels if we already did it
# (this way the notebook can be re-run without paying for Claude again)
LABELS_FILE = "../data/claude_labels.csv"
RELABEL = False   # set to True only if we want to run Claude again (about $0.60)

if RELABEL or not os.path.exists(LABELS_FILE):
    all_labels = []
    for start in range(0, len(neg), BATCH_SIZE):
        batch = neg["content"].iloc[start:start + BATCH_SIZE].tolist()
        try:
            all_labels += label_with_claude(batch)
        except Exception as e:
            print(f"Batch starting at {start} failed: {e}")
            all_labels += [["missing"]] * len(batch)
        if start % 500 == 0:
            print(f"Done {start + len(batch):,} of {len(neg):,}")

    # Saving the labels so we never have to run Claude again
    labels_df = pd.DataFrame({"reviewId": neg["reviewId"], "claude_labels": [json.dumps(l) for l in all_labels]})
    labels_df.to_csv(LABELS_FILE, index=False)
    cost = tokens["in"] / 1_000_000 * 1 + tokens["out"] / 1_000_000 * 5
    print(f"Saved! Total tokens (including the test): {tokens['in']:,} in, {tokens['out']:,} out | Total cost: ${cost:.2f}")
else:
    print("Loading saved labels from", LABELS_FILE)

# Adding the labels to our negative reviews
labels_df = pd.read_csv(LABELS_FILE)
labels_df["claude_labels"] = labels_df["claude_labels"].apply(json.loads)
neg = neg.merge(labels_df, on="reviewId")
print("Reviews with labels:", len(neg))
print("Reviews Claude skipped:", sum(labels == ["missing"] for labels in neg["claude_labels"]))

Done 20 of 4,147
Done 520 of 4,147
Done 1,020 of 4,147
Done 1,520 of 4,147
Done 2,020 of 4,147
Done 2,520 of 4,147
Done 3,020 of 4,147
Done 3,520 of 4,147
Done 4,020 of 4,147
Saved! Total tokens (including the test): 223,696 in, 74,803 out | Total cost: $0.60
Reviews with labels: 4147
Reviews Claude skipped: 0
